In [6]:
from complete_pipeline import CompletePipeline

pipeline = CompletePipeline(
    csv_path='../paper_verses.csv',
    output_dir='./results',
    num_perms=[128],
    shingle_sizes=[3, 4],
    thresholds=[0.1,0.2,0.75, 0.8, 0.85],
    poem_thresholds=[0.2, 0.3, 0.4, 0.5],
    n_jobs=-1
)

pipeline.run_full_pipeline()

2025-11-27 15:34:36,010 - INFO - 
System detected: 12 cores, 31 GB RAM
2025-11-27 15:34:36,011 - INFO - → Desktop mode: Using 11 cores
2025-11-27 15:34:36,011 - INFO - → Conservative memory mode: 10000 chunk size
2025-11-27 15:34:36,012 - INFO - ======================================================================
2025-11-27 15:34:36,012 - INFO - COMPLETE PRODUCTION PIPELINE
2025-11-27 15:34:36,012 - INFO - ======================================================================
2025-11-27 15:34:36,012 - INFO - Scratch: temp_files/complete_clustering
2025-11-27 15:34:36,013 - INFO - Verse grid search: 1×2×5 = 10 configs
2025-11-27 15:34:36,013 - INFO - Poem grid search: 4 thresholds
2025-11-27 15:34:36,013 - INFO - 
Step 1: Loading and validating data
2025-11-27 15:34:36,122 - WARNING - Removing 48 verses with NaN ground truth
2025-11-27 15:34:36,124 - INFO - Loaded 1,340 verses from 494 poems
2025-11-27 15:34:36,124 - INFO - Has verse ground truth: True
2025-11-27 15:34:36,125 - INFO -

## Print sample results

In [11]:
import pandas as pd
import numpy as np
df=pd.read_csv('results/verse_clusters.csv')
df=df.dropna()
def show_example_verse_clusters(df: pd.DataFrame, n_clusters: int = 3):
    clusters = (
        df['predicted_verse_cluster']
        .dropna()
        .astype(int)
        .unique()
    )
    cluster_sizes = (
    df.groupby("predicted_verse_cluster")
      .size()
      .rename("count")
    )

    clusters_with_multiple = cluster_sizes[cluster_sizes > 1].index.tolist()
    example_clusters = np.random.choice(clusters_with_multiple, size=min(n_clusters, len(clusters)), replace=True)
    
    for cid in example_clusters:
        print(f"Cluster {cid}")
        
        subset = df[df['predicted_verse_cluster'] == cid]
        
        for _, row in subset.iterrows():
            print(f"Verse: [Poem {row['idoriginal_poem']}]:  {row['verse']}")
        
        print("\n")
show_example_verse_clusters(df, n_clusters=10)

Cluster 775
Verse: [Poem 21112]:  πάσχοντας αἰσχρῶς ἐκ θεῶν ὁμοτρόπων
Verse: [Poem 23998]:  πάσχοντας αἰσχρῶς ἐκ θεῶν ὁμοτρόπ(ων)
Verse: [Poem 25029]:  πάσχοντας αἰσχρῶς ἐκ θεῶν ὁμοτρόπων


Cluster 775
Verse: [Poem 21112]:  πάσχοντας αἰσχρῶς ἐκ θεῶν ὁμοτρόπων
Verse: [Poem 23998]:  πάσχοντας αἰσχρῶς ἐκ θεῶν ὁμοτρόπ(ων)
Verse: [Poem 25029]:  πάσχοντας αἰσχρῶς ἐκ θεῶν ὁμοτρόπων


Cluster 775
Verse: [Poem 21112]:  πάσχοντας αἰσχρῶς ἐκ θεῶν ὁμοτρόπων
Verse: [Poem 23998]:  πάσχοντας αἰσχρῶς ἐκ θεῶν ὁμοτρόπ(ων)
Verse: [Poem 25029]:  πάσχοντας αἰσχρῶς ἐκ θεῶν ὁμοτρόπων




In [12]:
import pandas as pd

poems_df = pd.read_csv("results/poem_clusters.csv")
poems_df["idoriginal_poem"] = poems_df["idoriginal_poem"].astype(str)

cluster_sizes = poems_df.groupby("predicted_poem_cluster").size()
multi_poem_clusters = cluster_sizes[cluster_sizes > 1].index.tolist()
print(f"Found {len(multi_poem_clusters)} multi-poem clusters.")

clusters_to_show = multi_poem_clusters[:3]
poems_to_show = poems_df[poems_df["predicted_poem_cluster"].isin(clusters_to_show)]

cluster_to_poems = (
    poems_to_show.groupby("predicted_poem_cluster")["idoriginal_poem"]
    .apply(list)
    .to_dict()
)

verse_dict = {}  
chunksize = 100_000

for chunk in pd.read_csv("../paper_verses.csv", chunksize=chunksize):
    chunk["idoriginal_poem"] = chunk["idoriginal_poem"].astype(str)
    chunk = chunk[chunk["idoriginal_poem"].isin(poems_to_show["idoriginal_poem"])]
    
    for poem_id, group in chunk.groupby("idoriginal_poem"):
        verses = group.sort_values("order")["verse"].tolist()
        verses = [str(v) for v in verses if pd.notna(v)]
        if poem_id in verse_dict:
            verse_dict[poem_id].extend(verses)
        else:
            verse_dict[poem_id] = verses

for cl in clusters_to_show:
    print("\n" + "="*80)
    print(f"CLUSTER {cl}")
    print("="*80)
    
    for poem_id in cluster_to_poems[cl]:
        poem_text = "\n".join(verse_dict.get(poem_id, []))
        print(f"\n--- Poem {poem_id} ---\n")
        print(poem_text)
        print("\n" + "-"*40)


Found 28 multi-poem clusters.

CLUSTER 0

--- Poem 16907 ---

+ ὥσπερ ξένοι χαίρο(...) πατρίδα,
οὕτω (καὶ) οἱ γράφοντες τέλος βιβλίου +

----------------------------------------

--- Poem 17569 ---

ὥσπερ ξένοι χαίρ(...) | οἰδεῖν πατρίδα
(...)|τως καὶ ἡ γραφ(...) | τέλος βιβλί(...)

----------------------------------------

--- Poem 28311 ---

+ ὥσπερ ξένοι χαίροντ(...) ἰδεῖν πατρίδα 
οὕτω (καὶ) οἱ γράφοντες τέλος βιβλίου +

----------------------------------------

CLUSTER 4

--- Poem 16941 ---

+ Οσπερ ξένη χαίροντε ἠδὶν πατρηδαν
και οι θαλαττεύοντες ηδὶν λημένα 
και η στρατεβόμενοι ηδὶν τον ηκος
οντος και [..........] ηδὶν βιβλιου τελος.

----------------------------------------

--- Poem 31125 ---

+ Οσπερ ξένη χαίροντε ἠδὶν πατρηδαν
και οι θαλαττεύοντες ηδὶν λημένα 
και η στρατεβόμενοι ηδὶν τον ηκος
οντος και [..........] ηδὶν βιβλιου τελος.

----------------------------------------

CLUSTER 15

--- Poem 17188 ---

+ ἡ τετρὰς ὧδε τῶν μαθητῶν τοῦ λόγου:·
ἐκχεῖ τὸ ῥεῦμα τῶν ἀειρρύτω